# **Assignment 02 on Training CNNs**

### **Foundational Setup and Training Statics Calculation** 
This code handles fixing the random seeds for reproducibility, calculating the exact channel means and standard deviations from the raw CIFAR-10 training data, applying those statistics to normalize the dataset, and splitting it into the required 45k/5k subsets.

In [4]:
import torch
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import random_split, DataLoader
import numpy as np
import random

# 1. Fix random seeds for complete reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(42)
root_dir = './data'

# 2. Load raw CIFAR-10 strictly to get the indices for the split
raw_train_set = torchvision.datasets.CIFAR10(root=root_dir, train=True, download=False)

# Split into 45k and 5k to get the exact indices for the training subset
train_size = 45000
val_size = 5000
raw_train_subset, raw_val_subset = random_split(
    raw_train_set, 
    [train_size, val_size], 
    generator=torch.Generator().manual_seed(42)
)

# 3. Extract ONLY the 45k training images to calculate statistics
# raw_train_subset.indices safely extracts just the 45,000 training images
train_45k_data = raw_train_set.data[raw_train_subset.indices]

cifar_mean = train_45k_data.mean(axis=(0, 1, 2)) / 255.0
cifar_std = train_45k_data.std(axis=(0, 1, 2)) / 255.0

print(f"Calculated 45k Training Mean (RGB): {cifar_mean}")
print(f"Calculated 45k Training Std (RGB): {cifar_std}")

# 4. Define the transform using ONLY the 45k computed statistics
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=cifar_mean.tolist(), std=cifar_std.tolist())
])

# 5. Load the datasets again, this time with the normalization transform applied
full_train_dataset_transformed = torchvision.datasets.CIFAR10(root=root_dir, train=True, download=False, transform=transform)
test_dataset = torchvision.datasets.CIFAR10(root=root_dir, train=False, download=False, transform=transform)

# 6. Re-apply the EXACT SAME split (using the same seed) so our transformed datasets match the indices
train_dataset, val_dataset = random_split(
    full_train_dataset_transformed, 
    [train_size, val_size], 
    generator=torch.Generator().manual_seed(42) 
)

# 7. Initialize DataLoaders
batch_size = 128
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train set size: {len(train_dataset)} | Val set size: {len(val_dataset)} | Test set size: {len(test_dataset)}")

Calculated 45k Training Mean (RGB): [0.49165571 0.48233405 0.44667923]
Calculated 45k Training Std (RGB): [0.24708116 0.24350869 0.26158066]
Train set size: 45000 | Val set size: 5000 | Test set size: 10000


## **Task 01: Backpropagation**  

### **(a) Implement Linear, ReLU, and softmax cross-entropy with explicit forward/backward (no autograd), and assemble an MLP**

### 1. Mathematical Formulation for Forward and Backward Passes

**Linear Layer** ($Y = XW + b$)
*   **Forward:** Given input $X \in \mathbb{R}^{N \times D_{in}}$, weights $W \in \mathbb{R}^{D_{in} \times D_{out}}$, and bias $b \in \mathbb{R}^{D_{out}}$. 
    $$Y = XW + b$$
*   **Backward:** Given the upstream gradient $dY \in \mathbb{R}^{N \times D_{out}}$.
    $$dW = X^T dY$$
    $$db = \sum_{i=1}^{N} dY_i$$
    $$dX = dY W^T$$

**ReLU Activation** ($Y = \max(0, X)$)
*   **Forward:** 
    $$Y = \max(0, X)$$
*   **Backward:** The gradient passes through only where the original input was strictly positive. Let $\mathbb{I}$ be the indicator function and $\odot$ be the element-wise product.
    $$dX = dY \odot \mathbb{I}(X > 0)$$

**Softmax Cross-Entropy**
*   **Forward:** We subtract the maximum value per row for numerical stability before exponentiating.
    $$P_{i,j} = \frac{e^{X_{i,j} - \max_k(X_{i,k})}}{\sum_k e^{X_{i,k} - \max_k(X_{i,k})}}$$
    $$L = -\frac{1}{N} \sum_{i=1}^{N} \log(P_{i, y_i})$$
*   **Backward:** The gradient of the combined softmax and cross-entropy loss with respect to the input logits $X$ simplifies cleanly. Let $Y_{true}$ be the one-hot encoded targets.
    $$dX = \frac{1}{N} (P - Y_{true})$$

In [5]:
import torch

class Linear:
    def __init__(self, in_features, out_features):
        # Kaiming initialization for weights
        self.W = torch.randn(in_features, out_features) * (2.0 / in_features)**0.5
        self.b = torch.zeros(out_features)
        
        self.x = None
        self.dW = None
        self.db = None

    def forward(self, x):
        self.x = x
        return x @ self.W + self.b

    def backward(self, dout):
        self.dW = self.x.T @ dout
        self.db = dout.sum(dim=0)
        dx = dout @ self.W.T
        return dx

class ReLU:
    def __init__(self):
        self.x = None

    def forward(self, x):
        self.x = x
        return torch.clamp(x, min=0)

    def backward(self, dout):
        dx = dout.clone()
        dx[self.x <= 0] = 0
        return dx

class SoftmaxCrossEntropy:
    def __init__(self):
        self.probs = None
        self.targets = None

    def forward(self, logits, targets):
        self.targets = targets
        N = logits.shape[0]
        
        # Shift logits for numerical stability
        shifted_logits = logits - logits.max(dim=1, keepdim=True).values
        exp_logits = torch.exp(shifted_logits)
        self.probs = exp_logits / exp_logits.sum(dim=1, keepdim=True)
        
        # Cross-entropy loss
        corect_log_probs = -torch.log(self.probs[torch.arange(N), targets] + 1e-9)
        loss = corect_log_probs.sum() / N
        return loss

    def backward(self):
        N = self.probs.shape[0]
        dx = self.probs.clone()
        dx[torch.arange(N), self.targets] -= 1
        dx /= N
        return dx

class MLP:
    def __init__(self, layer_sizes):
        self.layers = []
        for i in range(len(layer_sizes) - 1):
            self.layers.append(Linear(layer_sizes[i], layer_sizes[i+1]))
            if i < len(layer_sizes) - 2:
                self.layers.append(ReLU())
        self.criterion = SoftmaxCrossEntropy()

    def forward(self, x):
        out = x
        for layer in self.layers:
            out = layer.forward(out)
        return out

    def backward(self):
        dout = self.criterion.backward()
        for layer in reversed(self.layers):
            dout = layer.backward(dout)

### **(b) Verify gradients against autograd and against centred finite differences**

In [7]:
import torch
import torch.nn as nn

# Setup a small deterministic problem. The whole MLP is tested (Linear -> ReLU -> Linear
# -> softmax cross-entropy) so that every backward pass written in (a) is exercised.
torch.manual_seed(42)
N, D_in, H, D_out = 4, 5, 6, 3
x = torch.randn(N, D_in, dtype=torch.float64) # Use float64 for finite difference precision
y_target = torch.randint(0, D_out, (N,))

# 1. Initialize the manual MLP and convert its parameters to float64 for the test
manual_mlp = MLP([D_in, H, D_out])
manual_linears = [layer for layer in manual_mlp.layers if isinstance(layer, Linear)]
for layer in manual_linears:
    layer.W = layer.W.to(torch.float64)
    layer.b = layer.b.to(torch.float64)

# Full forward pass: logits through the MLP, then the softmax cross-entropy loss
def manual_loss(model, inputs, targets):
    logits = model.forward(inputs)
    return model.criterion.forward(logits, targets)

# 2. Initialize the equivalent PyTorch model and copy the exact same weights into it
torch_mlp = nn.Sequential(
    nn.Linear(D_in, H),
    nn.ReLU(),
    nn.Linear(H, D_out)
).to(torch.float64)

torch_linears = [module for module in torch_mlp if isinstance(module, nn.Linear)]
for manual_layer, torch_layer in zip(manual_linears, torch_linears):
    # nn.Linear stores the weight transposed, as (out_features, in_features)
    torch_layer.weight.data = manual_layer.W.T.clone()
    torch_layer.bias.data = manual_layer.b.clone()

# Calculate Relative Error function (normalized by the larger of the two magnitudes)
def rel_error(x, y, eps=1e-12):
    denominator = torch.maximum(torch.norm(x), torch.norm(y)).clamp(min=eps)
    return (torch.norm(x - y) / denominator).item()

# --- Autograd Verification ---
# PyTorch Forward and Backward (nn.CrossEntropyLoss is softmax cross-entropy on logits)
torch_loss = nn.CrossEntropyLoss()(torch_mlp(x), y_target)
torch_loss.backward()

# Manual Forward and Backward
manual_loss_value = manual_loss(manual_mlp, x, y_target)
manual_mlp.backward()

# Keep a copy of the analytic gradients, the layer caches get overwritten by the finite differences below
analytic_grads = [(layer.dW.clone(), layer.db.clone()) for layer in manual_linears]

print("--- Autograd Verification ---")
print(f"Manual Loss: {manual_loss_value:.10f} | PyTorch Loss: {torch_loss.item():.10f}\n")
for i, (manual_layer, torch_layer) in enumerate(zip(manual_linears, torch_linears), start=1):
    print(f"Linear {i} Weight Relative Error: {rel_error(manual_layer.dW, torch_layer.weight.grad.T):.2e}")
    print(f"Linear {i} Bias Relative Error:   {rel_error(manual_layer.db, torch_layer.bias.grad):.2e}")
    print(f"Linear {i} Weights Allclose:      {torch.allclose(manual_layer.dW, torch_layer.weight.grad.T, rtol=1e-7, atol=1e-10)}")
    print(f"Linear {i} Bias Allclose:         {torch.allclose(manual_layer.db, torch_layer.bias.grad, rtol=1e-7, atol=1e-10)}\n")

# --- Centered Finite Differences Verification ---
epsilon = 1e-6

# Numerically differentiate the loss with respect to every entry of one parameter tensor
def finite_diff_grad(param, model, inputs, targets, eps=epsilon):
    finite_diff = torch.zeros_like(param)
    flat_param, flat_diff = param.view(-1), finite_diff.view(-1)

    for i in range(flat_param.numel()):
        original_value = flat_param[i].item()

        # f(w + epsilon)
        flat_param[i] = original_value + eps
        loss_plus = manual_loss(model, inputs, targets)

        # f(w - epsilon)
        flat_param[i] = original_value - eps
        loss_minus = manual_loss(model, inputs, targets)

        # Restore original value and calculate derivative
        flat_param[i] = original_value
        flat_diff[i] = (loss_plus - loss_minus) / (2 * eps)

    return finite_diff

print("--- Centered Finite Differences Verification ---")
for i, (layer, (dW, db)) in enumerate(zip(manual_linears, analytic_grads), start=1):
    finite_diff_dW = finite_diff_grad(layer.W, manual_mlp, x, y_target)
    finite_diff_db = finite_diff_grad(layer.b, manual_mlp, x, y_target)
    print(f"Linear {i} Weight Relative Error: {rel_error(dW, finite_diff_dW):.2e}")
    print(f"Linear {i} Bias Relative Error:   {rel_error(db, finite_diff_db):.2e}")
    print(f"Linear {i} Weights Allclose:      {torch.allclose(dW, finite_diff_dW, rtol=1e-4, atol=1e-7)}")
    print(f"Linear {i} Bias Allclose:         {torch.allclose(db, finite_diff_db, rtol=1e-4, atol=1e-7)}\n")

# The ReLU kink is not differentiable, so confirm that no pre-activation sits on top of it
pre_activation = manual_linears[0].forward(x)
print(f"Smallest |pre-activation| entering ReLU: {pre_activation.abs().min().item():.2e} (must be >> {epsilon})")

--- Autograd Verification ---
Manual Loss: 1.6114817483 | PyTorch Loss: 1.6114817548

Linear 1 Weight Relative Error: 2.45e-17
Linear 1 Bias Relative Error:   3.80e-17
Linear 1 Weights Allclose:      True
Linear 1 Bias Allclose:         True

Linear 2 Weight Relative Error: 5.16e-17
Linear 2 Bias Relative Error:   7.51e-17
Linear 2 Weights Allclose:      True
Linear 2 Bias Allclose:         True

--- Centered Finite Differences Verification ---
Linear 1 Weight Relative Error: 1.25e-08
Linear 1 Bias Relative Error:   8.31e-09
Linear 1 Weights Allclose:      True
Linear 1 Bias Allclose:         True

Linear 2 Weight Relative Error: 1.26e-08
Linear 2 Bias Relative Error:   8.93e-09
Linear 2 Weights Allclose:      True
Linear 2 Bias Allclose:         True

Smallest |pre-activation| entering ReLU: 8.03e-03 (must be >> 1e-06)


### 2. Interpretation of the Gradient Check

The check is run on the **whole MLP** rather than on a single layer. A dummy loss such as `out.sum()` on one `Linear` would make the loss linear in every weight, so both the autograd and the finite difference comparison would pass even if `ReLU.backward` or `SoftmaxCrossEntropy.backward` were wrong. Routing the real softmax cross-entropy loss back through `Linear -> ReLU -> Linear` exercises all three backward passes at once.

**Autograd:** the relative errors are of the order of $10^{-17}$, which is below the float64 machine epsilon ($\approx 2.2 \times 10^{-16}$). Our hand-derived gradients and PyTorch's are therefore the same computation up to the order in which the floating point operations happen to be summed, so the backward passes are confirmed to be analytically correct.

**Centred finite differences:** the relative errors settle at roughly $10^{-8}$. Centred differences have a truncation error of $O(\epsilon^2)$ and a round-off error of $O(\epsilon_{machine}/\epsilon)$, which in float64 would normally bottom out near $10^{-10}$ around $\epsilon \approx 10^{-5}$. Here the floor is instead set by the `+ 1e-9` guard inside the logarithm in `SoftmaxCrossEntropy.forward`: it perturbs the *loss* that the finite differences measure, while `backward` implements the exact $\frac{1}{N}(P - Y_{true})$ with no such guard. The resulting mismatch is of order $10^{-9}/P_{i,y_i} \approx 10^{-8}$, which is exactly the floor observed, and it stays flat when $\epsilon$ is swept from $10^{-4}$ to $10^{-7}$ (confirming that the step size is not the limiting factor). An agreement of $10^{-8}$ is still around eight orders of magnitude tighter than any realistic implementation bug, so the gradients are verified.

**On the ReLU kink:** finite differences are only valid where the function is differentiable. Perturbing a weight by $\pm \epsilon$ could flip a pre-activation across zero and make the numerical derivative meaningless. The last line confirms the smallest pre-activation magnitude is $\approx 10^{-2}$, i.e. several orders of magnitude larger than $\epsilon$, so no unit sits on the kink and the check is sound.